# Anti-Spoofing Detection using SigLIP

Notebook ini menyediakan pipeline end-to-end untuk melatih model deteksi anti-spoofing (liveness detection) menggunakan model **SigLIP** (Signal Language-Image Pre-training) dari Google. 

Karena dataset berada di Google Drive, notebook ini sudah dilengkapi dengan cell untuk melakukan mounting Google Drive.

## 1. Setup Environment
Instalasi library yang dibutuhkan seperti `transformers` untuk memuat model SigLIP.

In [ ]:
!pip install -q transformers torch torchvision scikit-learn tqdm matplotlib seaborn pillow

## 2. Mount Google Drive
Jalankan cell di bawah ini untuk menghubungkan Google Colab dengan Google Drive Anda.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# TODO: Ubah path ini sesuai dengan lokasi folder dataset anti-spoofing Anda di Google Drive
DATA_DIR = '/content/drive/MyDrive/Dataset_AntiSpoofing'

# Asumsi struktur folder dataset:
# DATA_DIR/
# ├── real/
# │   ├── img1.jpg
# │   └── ...
# └── spoof/
#     ├── img2.jpg
#     └── ...

## 3. Import Libraries

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import AutoProcessor, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

## 4. Persiapan Dataset (Dataloader)

In [ ]:
class AntiSpoofingDataset(Dataset):
    def __init__(self, file_paths, labels, processor):
        self.file_paths = file_paths
        self.labels = labels
        self.processor = processor

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        image_path = self.file_paths[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            image = Image.new("RGB", (224, 224))
            
        # Processor SigLIP memproses gambar (resize, normalize, dll)
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0) # Hapus dimensi batch
        
        return pixel_values, torch.tensor(label, dtype=torch.long)

def prepare_data(data_dir):
    file_paths = []
    labels = []
    
    # Label: 0 untuk Real (Asli), 1 untuk Spoof (Palsu)
    classes = {"real": 0, "spoof": 1}
    
    for class_name, label in classes.items():
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.exists(class_dir):
            print(f"Peringatan: Direktori {class_dir} tidak ditemukan!")
            continue
            
        for img_name in os.listdir(class_dir):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                file_paths.append(os.path.join(class_dir, img_name))
                labels.append(label)
                
    return file_paths, labels

## 5. Arsitektur Model (SigLIP + Classifier)
Kita akan menggunakan pretrained image encoder dari SigLIP sebagai feature extractor dan menambahkan lapisan klasifikasi sederhana di atasnya.

In [ ]:
class SigLIPAntiSpoofingModel(nn.Module):
    def __init__(self, model_name="google/siglip-base-patch16-224", num_classes=2, freeze_encoder=True):
        super().__init__()
        # Memuat pretrained SigLIP
        self.siglip = AutoModel.from_pretrained(model_name)
        
        # Freeze encoder agar kita hanya melatih head klasifikasinya (transfer learning)
        if freeze_encoder:
            for param in self.siglip.parameters():
                param.requires_grad = False
                
        # Hidden size standar SigLIP-base adalah 768
        hidden_size = self.siglip.config.vision_config.hidden_size
        
        # Custom classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, pixel_values):
        # Ekstrak fitur dari model vision SigLIP
        outputs = self.siglip.vision_model(pixel_values=pixel_values)
        pooled_output = outputs.pooler_output
        
        # Prediksi
        logits = self.classifier(pooled_output)
        return logits

## 6. Konfigurasi dan Dataloader Setup

In [ ]:
# Konfigurasi
MODEL_NAME = "google/siglip-base-patch16-224"
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 3e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Menggunakan device: {DEVICE}")

# Inisialisasi processor
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# Membaca dataset
file_paths, labels = prepare_data(DATA_DIR)
print(f"Total gambar ditemukan: {len(file_paths)}")

if len(file_paths) > 0:
    # Membagi dataset menjadi Train (80%) dan Validation (20%)
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        file_paths, labels, test_size=0.2, random_state=42, stratify=labels
    )
    
    print(f"Jumlah Train: {len(train_paths)} | Jumlah Validasi: {len(val_paths)}")

    train_dataset = AntiSpoofingDataset(train_paths, train_labels, processor)
    val_dataset = AntiSpoofingDataset(val_paths, val_labels, processor)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
else:
    print("TIDAK ADA DATA! Pastikan DATA_DIR sudah benar dan dataset sudah ada di Google Drive.")

## 7. Training Loop

In [ ]:
# Inisialisasi model, loss function, dan optimizer
model = SigLIPAntiSpoofingModel(model_name=MODEL_NAME, freeze_encoder=True).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=LEARNING_RATE)

best_val_acc = 0.0
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(EPOCHS):
    # --- Fase Training ---
    model.train()
    running_loss = 0.0
    train_preds, train_targets = [], []
    
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for pixel_values, targets in train_bar:
        pixel_values, targets = pixel_values.to(DEVICE), targets.to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(pixel_values)
        loss = criterion(logits, targets)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        train_preds.extend(preds.cpu().numpy())
        train_targets.extend(targets.cpu().numpy())
        
        train_bar.set_postfix(loss=loss.item())
        
    epoch_train_acc = accuracy_score(train_targets, train_preds)
    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)
    train_accs.append(epoch_train_acc)
    
    # --- Fase Validasi ---
    model.eval()
    running_val_loss = 0.0
    val_preds, val_targets = [], []
    
    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]")
        for pixel_values, targets in val_bar:
            pixel_values, targets = pixel_values.to(DEVICE), targets.to(DEVICE)
            
            logits = model(pixel_values)
            loss = criterion(logits, targets)
            
            running_val_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(targets.cpu().numpy())
            
    epoch_val_acc = accuracy_score(val_targets, val_preds)
    epoch_val_loss = running_val_loss / len(val_loader)
    val_losses.append(epoch_val_loss)
    val_accs.append(epoch_val_acc)
    
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
    
    # Simpan model terbaik
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), "best_siglip_antispoofing.pth")
        print("--> Model terbaik disimpan (best_siglip_antispoofing.pth)!")

print("Training Selesai!")

## 8. Evaluasi (Kurva & Confusion Matrix)

In [ ]:
# Plot Loss dan Accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses, label='Val Loss')
ax1.set_title('Training & Validation Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(train_accs, label='Train Acc')
ax2.plot(val_accs, label='Val Acc')
ax2.set_title('Training & Validation Accuracy')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.show()

print("\nClassification Report:")
print(classification_report(val_targets, val_preds, target_names=["Real", "Spoof"]))

# Confusion Matrix
cm = confusion_matrix(val_targets, val_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Real", "Spoof"], yticklabels=["Real", "Spoof"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

## 9. Inference (Prediksi Gambar Baru)
Fungsi untuk melakukan prediksi pada gambar baru apakah Real atau Spoof.

In [ ]:
def predict_image(image_path, model, processor, device):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    
    inputs = processor(images=image, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)
    
    with torch.no_grad():
        logits = model(pixel_values)
        prob = torch.softmax(logits, dim=1)
        pred_class = torch.argmax(prob, dim=1).item()
        confidence = prob[0][pred_class].item()
        
    classes = {0: "Real (Asli)", 1: "Spoof (Palsu)"}
    
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"Prediksi: {classes[pred_class]} ({confidence:.2%} Yakin)")
    plt.show()

# Contoh penggunaan (Pastikan path file ini benar):
# model.load_state_dict(torch.load("best_siglip_antispoofing.pth"))
# sample_image_path = "/content/drive/MyDrive/Dataset_AntiSpoofing/real/sample_img.jpg"
# predict_image(sample_image_path, model, processor, DEVICE)